# Store article embeddings in ChromaDB

Store article-level SEA-LION and Granite R2 embeddings in separate ChromaDB collections.

In [1]:
import json
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
EMBEDDING_DIR = DATA_DIR / 'processed' / 'embeddings'
CHROMA_DIR = DATA_DIR / 'processed' / 'chroma_db'
BATCH_SIZE = 256

DATASETS = {
    'sealion': ('articles_sealion_e5_600m', 'legal_articles_sealion'),
    'granite-r2': ('articles_granite_311m_r2', 'legal_articles_granite_r2'),
}

def load_dataset(prefix, level):
    vector_path = EMBEDDING_DIR / f'{prefix}.npy'
    metadata_path = EMBEDDING_DIR / f'{prefix}_metadata.jsonl'
    manifest_path = EMBEDDING_DIR / f'{prefix}_manifest.json'
    missing = [str(path) for path in (vector_path, metadata_path, manifest_path) if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing embedding output: ' + ', '.join(missing))
    vectors = np.load(vector_path).astype('float32')
    records = [json.loads(line) for line in metadata_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected_count = manifest.get('article_count', manifest.get('chunk_count'))
    if vectors.ndim != 2 or len(records) != vectors.shape[0] or expected_count != vectors.shape[0]:
        raise ValueError(f'{level} vectors, metadata, and manifest counts do not match')
    if manifest.get('embedding_dimension') != vectors.shape[1]:
        raise ValueError(f'{level} manifest dimension does not match vectors')
    return vectors, records, manifest

def scalar_metadata(record, model_name):
    result = {'embedding_model': model_name}
    for key, value in record.items():
        if key in {'text', 'search_text'} or value is None:
            continue
        result[key] = value if isinstance(value, (str, int, float, bool)) else json.dumps(value, ensure_ascii=False)
    return result

def store_dataset(model_key):
    import chromadb
    prefix, collection_name = DATASETS[model_key]
    vectors, records, manifest = load_dataset(prefix, 'article')
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    collection = client.get_or_create_collection(name=collection_name, metadata={'model': manifest['model'], 'hnsw:space': 'cosine'})
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        ids = [str(record['id']) for record in batch]
        collection.upsert(ids=ids, embeddings=vectors[start:start + BATCH_SIZE].tolist(), documents=[str(record.get('text', '')) for record in batch], metadatas=[scalar_metadata(record, manifest['model']) for record in batch])
    return {'level': 'article', 'model': model_key, 'collection': collection_name, 'stored': collection.count(), 'dimension': int(vectors.shape[1])}

print('ChromaDB:', CHROMA_DIR)
print('Datasets:', DATASETS)

ChromaDB: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/chroma_db
Datasets: {'article': {'sealion': ('articles_sealion_e5_600m', 'legal_articles_sealion'), 'granite-r2': ('articles_granite_311m_r2', 'legal_articles_granite_r2')}, 'chunk': {'sealion': ('chunks_sealion_e5_600m', 'legal_chunks_sealion'), 'granite-r2': ('chunks_granite_311m_r2', 'legal_chunks_granite_r2')}}


In [2]:
results = []
for model_key in DATASETS:
    results.append(store_dataset(model_key))

print(json.dumps(results, ensure_ascii=False, indent=2))

[
  {
    "level": "article",
    "model": "sealion",
    "collection": "legal_articles_sealion",
    "stored": 2744,
    "dimension": 1024
  },
  {
    "level": "article",
    "model": "granite-r2",
    "collection": "legal_articles_granite_r2",
    "stored": 2744,
    "dimension": 768
  },
  {
    "level": "chunk",
    "model": "sealion",
    "collection": "legal_chunks_sealion",
    "stored": 3034,
    "dimension": 1024
  },
  {
    "level": "chunk",
    "model": "granite-r2",
    "collection": "legal_chunks_granite_r2",
    "stored": 3034,
    "dimension": 768
  }
]
